# Activation functions and ribosome allocation

Simulations supporting Figure 2 and the supplementary analyses of activation functions and resource-competition onset.


In [1]:
# Colab setup
import os, sys, subprocess
if "google.colab" in sys.modules:
  cmd = "pip install --upgrade biocircuits bokeh-catplot watermark blackcellmagic intersect"
  process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  stdout, stderr = process.communicate()

# ------ Solving ODEs
import scipy.integrate
import biocircuits

# ------ Steady-state analysis
import numpy as np
from intersect import intersection
from skimage import measure
import sympy as sym
import pylab as pl

# ------ Visualization
import bokeh.io
import bokeh.plotting
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d, Arrow, OpenHead
from bokeh.io import export_svgs

bokeh.io.output_notebook()

Loading BokehJS ...

In [2]:
def Seq_TL_ODEs(X, t, x1, x2, x3, ap1, ap2, ap3, p, d, a1, a2, a3, d1, d2, d3,\
                 v1, v2, v3, rtot, b, u):

  m1, m2, m3, r1, r2, r3, y1, y2, y3, c = X

  r = rtot - (r1 + r2 + r3)

  return (
      [
          # Transcription - Translation (TXTL)
          ap1*(x1) - p*m1 - d*m1 - a1*m1*r + d1*r1 + v1*r1,
          ap2*(x2) - p*m2 - d*m2 - a2*m2*r + d2*r2 + v2*r2,
          ap3*(x3) - p*m3 - d*m3 - a3*m3*r + d3*r3 + v3*r3,
          a1*m1*r - d1*r1 - v1*r1 - d*r1 - p*r1,
          a2*m2*r - d2*r2 - v2*r2 - d*r2 - p*r2,
          a3*m3*r - d3*r3 - v3*r3 - d*r3 - p*r3,

          # Molecular sequestration
          v1*r1 - d*y1 - b*y1*y2 + u*c,
          v2*r2 - d*y2 - b*y1*y2 + u*c,
          v3*r3 - d*y3,
          b*y1*y2 - u*c - d*c
      ]
  )

## Fast-sequestration approximation under abundant resources

This simulations sets the baseline on the geometry of the activation function for abundant resources ($r^{tot} = 10 \, \mu$M) and no endogenous demand ($x_e = x_3 = 0 \, \mu$M).


In [3]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Inputs
xv = np.linspace(0, 1.5, 80)
x2 = 0.5
x3 = 0.0   # no endogenous demand

# Transcription rates (μM/hr): 1 μM ≈ 600 molecules per E. coli cell
# 10 μM/hr ≈ 6,000 transcripts per cell per hour
# ≈ 100 transcripts per plasmid per hour for ~60-copy plasmid
# Then, 10.0 represents a strong promoter × medium-high copy number
ap1 = 10.0
ap2 = 10.0
ap3 = 10.0

# Degradation rates
p = 10.0     # hr⁻¹, mRNA decay (t½ ≈ 4.2 min, typical E. coli)
d = 1.0      # hr⁻¹, dilution by growth (t_double ≈ 60 min)

# Ribosome binding
# Kd = d1 / a = 1 / 100 = 0.01 μM = 10 nM (i.e., strong RBS)
a1 = a2 = a3 = 100.0   # μM⁻¹·hr⁻¹, effective ribosome–mRNA association
d1 = d2 = d3 = 1.0     # hr⁻¹, effective dissociation

# Translation: effective protein production per mRNA–ribosome complex
v1 = v2 = v3 = 1.0     # hr⁻¹, ~1 protein per complex per hour

# Sequestration
# ξ = (d + u) / b = 2 / 200 = 0.01 μM = 10 nM (strong protein–protein binding)
b = 200.0    # μM⁻¹·hr⁻¹
u = 1.0      # hr⁻¹ (t½ ≈ 42 min)

# Total ribosome pool
# 1 μM ≈ 600 ribosomes per cell
rtot = 10   # μM

# Sweeping b(inding) values (large b → small ξ)
b_values = [10.0, 50.0, 200.0, 2000.0]
xi_values = [(d + u) / b for b in b_values]

# Compute the activation function curve
output_curves = []
y0 = np.zeros(10)

for b_val in b_values:
    output = np.zeros_like(xv)
    for i, xi_input in enumerate(xv):
        sol = scipy.integrate.odeint(
            Seq_TL_ODEs, y0, t,
            args=(xi_input, x2, x3,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot, b_val, u)
        )
        output[i] = sol[-1, 6]
    output_curves.append(output)

# Plotting
fig_size = [250, 225]
plot = bokeh.plotting.figure(
    width=fig_size[0], height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁ (μM)'
)
plot.axis.major_label_text_font_size = '14pt'
plot.xaxis.axis_label_text_font_size = '14pt'
plot.yaxis.axis_label_text_font_size = '14pt'

# Color palette
colors = bokeh.palettes.Greys8[::-1]

for i, output in enumerate(output_curves):
    plot.line(xv, output,
                line_width=3.5,
                color=colors[i+3])

# Analytical ReLU
w1 = (ap1 * v1) / ((d + p) * d)
w2 = (ap2 * v2) / ((d + p) * d)
relu_ref = np.maximum(0.0, w1*xv - w2*x2)
plot.line(xv, relu_ref,
            line_width=1.5, color='orange', line_dash='dashed')

plot.output_backend = 'svg'
bokeh.io.show(plot)

## Activation function under increasing endogenous demand


In [4]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Inputs
xv = np.linspace(0, 3, 51)
x2 = 0.5

# Transcription rates (μM/hr): 1 μM ≈ 600 molecules per E. coli cell
# 10 μM/hr ≈ 6,000 transcripts per cell per hour
# ≈ 100 transcripts per plasmid per hour for ~60-copy plasmid
# Then, 10.0 represents a strong promoter × medium-high copy number
ap1 = 10.0
ap2 = 10.0
ap3 = 10.0

# Degradation rates
p = 10.0     # hr⁻¹, mRNA decay (t½ ≈ 4.2 min, typical E. coli)
d = 1.0      # hr⁻¹, dilution by growth (t_double ≈ 60 min)

# Ribosome binding
# Kd = d1 / a = 1 / 100 = 0.01 μM = 10 nM (i.e., strong RBS)
a1 = a2 = a3 = 100.0   # μM⁻¹·hr⁻¹, effective ribosome–mRNA association
d1 = d2 = d3 = 1.0     # hr⁻¹, effective dissociation

# Translation: effective protein production per mRNA–ribosome complex
v1 = v2 = v3 = 1.0     # hr⁻¹, ~1 protein per complex per hour

# Sequestration
# Kd = u / b = 1 / 200 = 0.005 μM = 5 nM (strong protein–protein binding)
b = 200.0    # μM⁻¹·hr⁻¹
u = 1.0      # hr⁻¹ (t½ ≈ 42 min)

# Total ribosome pool
# 1 μM ≈ 600 ribosomes per cell
rtot = 1   # μM

# Sweep x3
n_curves = 6
x3_values = np.linspace(0.1, 1.0, n_curves)

# Compute input-output curves (activation function)
output_curves = []
y0 = np.zeros(10)

for x3_val in x3_values:
    output = np.zeros_like(xv)

    for i, xi in enumerate(xv):
        sol = scipy.integrate.odeint(
            Seq_TL_ODEs, y0, t,
            args=(xi, x2, x3_val,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot, b, u)
        )

        output[i] = sol[-1, 6]

    output_curves.append(output)

# Plotting
fig_size = [250, 225]

plot_A = bokeh.plotting.figure(
    width=fig_size[0],
    height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁ (μM)',
)

plot_A.axis.major_label_text_font_size = '14pt'
plot_A.xaxis.axis_label_text_font_size = '14pt'
plot_A.yaxis.axis_label_text_font_size = '14pt'

colors = bokeh.palettes.Greys8[::-1]

# Threshold set by x2
# yv = bokeh.models.Span(location=0.5, dimension='height', line_color='orange', line_width=2,
#                        line_dash = 'dashed')
# plot_A.add_layout(yv)

for i, (output, x3_val) in enumerate(zip(output_curves, x3_values)):
    plot_A.line(xv, output,
                line_width=2.5,
                color=colors[i+2])

plot_A.output_backend = 'svg'
bokeh.io.show(plot_A)

## Output attenuation under changes in endogenous binding affinity and ribosome supply


In [5]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Inputs
xv = np.linspace(0, 3, 51)
x2 = 0.5
x3 = 1.0   # fixed endogenous demand

# Transcription rates (μM/hr)
ap1 = 10.0
ap2 = 10.0
ap3 = 10.0

# Degradation rates
p = 10.0     # hr⁻¹, mRNA decay (t½ ≈ 4.2 min, typical E. coli)
d = 1.0      # hr⁻¹, dilution by growth (t_double ≈ 60 min)

# Ribosome binding for the synthetic circuit (realistic, Kd = 10 nM)
a1 = a2 = 100.0        # μM⁻¹·hr⁻¹
d1 = d2 = d3 = 1.0     # hr⁻¹

# Translation
v1 = v2 = v3 = 1.0     # hr⁻¹

# Sequestration
b = 200.0
u = 1.0

# Strong-binding synthetic circuit (used across both panels)
e_val = 1e-6

# ============================================================================
# Panel A1: sweep e3 (endogenous binding affinity) at fixed rtot
# ============================================================================

# Total ribosome pool (fixed for this sweep)
rtot = 1.0   # μM

# Sweep e3 (endogenous mRNA-ribosome affinity)
# Strong endogenous → larger a3, more ribosome consumption
n_curves = 6
e3_values = np.logspace(-6, -9, n_curves)   # 1e-6 (weakest) → 1e-9 (strongest)

# Compute input-output curves (activation function)
output_curves_e3 = []
y0 = np.zeros(10)

for e3_val in e3_values:
    a3 = (d3 + v3) / e3_val
    a1 = (d1 + v1) / e_val
    a2 = (d2 + v2) / e_val
    output = np.zeros_like(xv)
    for i, xi in enumerate(xv):
        sol = scipy.integrate.odeint(
            Seq_TL_ODEs, y0, t,
            args=(xi, x2, x3,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot, b, u)
        )
        output[i] = sol[-1, 6]
    output_curves_e3.append(output)

# Plotting
fig_size = [250, 225]

plot_A1 = bokeh.plotting.figure(
    width=fig_size[0],
    height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁ (μM)',
)

plot_A1.axis.major_label_text_font_size = '14pt'
plot_A1.xaxis.axis_label_text_font_size = '14pt'
plot_A1.yaxis.axis_label_text_font_size = '14pt'

colors = bokeh.palettes.Greys8[::-1]   # light → dark

# Plot in the natural order of e3_values (weakest → strongest binding).
# Weakest binding → highest output → lightest shade;
# strongest binding → lowest output → darkest shade.
for i, (output, e3_val) in enumerate(zip(output_curves_e3, e3_values)):
    plot_A1.line(xv, output,
                 line_width=2.5,
                 color=colors[i+2],
                 #legend_label="e3 = {:.2e}".format(e3_val)
                 )

plot_A1.output_backend = 'svg'

# ============================================================================
# Panel A2: sweep rtot at fixed (weak) e3 and x3
# ============================================================================

# Restore synthetic-circuit binding (in case Panel A1's loop was reordered)
a1 = (d1 + v1) / e_val
a2 = (d2 + v2) / e_val

# Fixed endogenous binding (weak) and demand for this sweep
e3_fixed = 100.0
a3 = (d3 + v3) / e3_fixed

# Sweep rtot (total ribosome pool)
# Smaller rtot → more competition for the synthetic circuit
rtot_values = np.linspace(0.5, 3.0, n_curves)
rtot_values = rtot_values[::-1]   # 3.0 (largest) → 0.5 (smallest)

# Compute input-output curves (activation function)
output_curves_rtot = []

for rtot_val in rtot_values:
    output = np.zeros_like(xv)
    for i, xi in enumerate(xv):
        sol = scipy.integrate.odeint(
            Seq_TL_ODEs, y0, t,
            args=(xi, x2, x3,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot_val, b, u)
        )
        output[i] = sol[-1, 6]
    output_curves_rtot.append(output)

# Plotting
plot_A2 = bokeh.plotting.figure(
    width=fig_size[0],
    height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁ (μM)',
)

plot_A2.axis.major_label_text_font_size = '14pt'
plot_A2.xaxis.axis_label_text_font_size = '14pt'
plot_A2.yaxis.axis_label_text_font_size = '14pt'

# Plot in the current order of rtot_values (largest → smallest).
# Largest pool → highest output → lightest shade;
# smallest pool → lowest output → darkest shade.
for i, (output, rtot_val) in enumerate(zip(output_curves_rtot, rtot_values)):
    plot_A2.line(xv, output,
                 line_width=2.5,
                 color=colors[i+2],
                 #legend_label="rtot = {}".format(rtot_val)
                 )

plot_A2.output_backend = 'svg'

# Show both panels side by side
bokeh.io.show(bokeh.layouts.row(plot_A1, plot_A2))

## Ribosome allocation under homogeneous binding affinities

Plot $S$ vs $R^*/\Omega$ for several binding affinities, with the analytical limit $\min(1, R^*/\Omega)$ overlaid. Sweep $x_1$ to vary $R^*/\Omega$; $x_3$ fixed. As $e \to 0$, the curves collapse onto the analytical kink at $R^*/\Omega = 1$.


In [6]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Inputs
x1_sweep = np.linspace(0.01, 6.0, 80)
x2 = 0.5
x3 = 1.0   # fixed endogenous demand

# Color palette (strong → weak binding)
colors = ['#000000', '#444444', '#BBBBBB', '#E8E8E8']

# Kinetic rates
ap1 = 10.0              # (μM/hr)
ap2 = 10.0              # (μM/hr)
ap3 = 10.0              # (μM/hr)
p = 10.0                # hr⁻¹
d = 1.0                 # hr⁻¹
d1 = d2 = d3 = 1.0      # hr⁻¹
v1 = v2 = v3 = 1.0      # hr⁻¹
b = 200.0               # μM⁻¹·hr⁻¹
u = 1.0                 # hr⁻¹
rtot = 2                # μM

# Endogenous species binding affinity (fixed, weak)
# Manuscript convention: e_i = (d_i + v_i + δ + ϕ) / (a_i · r^tot)
# so a_i = (d_i + v_i + δ + ϕ) / e_i  with e_i taken to be the dimensionless e
e3 = 100
a3 = (d3 + v3 + d + p) / e3

# Binding affinity for the synthetic circuit (strong → weak)
e_values = [1e-6, 1e-3, 1e-2, 1e-1]

# Figure setup
fig_size = [250, 225]
plot_B1 = bokeh.plotting.figure(
    width=fig_size[0], height=fig_size[1],
    x_axis_label='R*/Ω',
    y_axis_label='S',
    #x_range=(0, 3.5),
    y_range=(0.2, 1.05)
)
plot_B1.axis.major_label_text_font_size = '12pt'
plot_B1.xaxis.axis_label_text_font_size = '14pt'
plot_B1.yaxis.axis_label_text_font_size = '14pt'

# Analytical Eq. (12): S = min(1, R*/Omega)
xx = np.linspace(0.01, 3.5, 200)
plot_B1.line(xx, np.minimum(1, xx),
            line_width=2, color='orange', line_dash='dashed')

# Compute S vs R*/Omega for each binding affinity
for e_val, color in zip(e_values, colors):
    # Manuscript convention: a = (d + v + δ + ϕ) / e
    a1 = (d1 + v1 + d + p) / e_val
    a2 = (d2 + v2 + d + p) / e_val

    ratio_arr = []
    S_arr = []
    for xi in x1_sweep:
        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs, np.zeros(10), t,
            args=(xi, x2, x3,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot, b, u)
        )
        # Extract ribosome concentrations at steady state
        r1_ss = y_sol[-1, 3]
        r2_ss = y_sol[-1, 4]
        r3_ss = y_sol[-1, 5]
        r_free = rtot - (r1_ss + r2_ss + r3_ss)
        r_n = r_free / rtot

        if r_n <= 1e-6:
            continue

        # S for the synthetic circuit (homogeneous, so S1 = S2 = S)
        S = r_n / (r_n + e_val)

        # Demand from synthetic circuit
        Omega = (ap1*xi + ap2*x2) / ((d + p) * rtot)

        # Effective ribosome availability (with endogenous demand)
        Omega3 = (ap3 * x3) / ((d + p) * rtot)
        S3 = r_n / (r_n + e3)
        Rstar = 1.0 - S3 * Omega3

        if Omega > 0:
            ratio_arr.append(Rstar / Omega)
            S_arr.append(S)

    # Sort by R*/Omega for clean plotting
    ratio_arr = np.array(ratio_arr)
    S_arr = np.array(S_arr)
    order = np.argsort(ratio_arr)

    plot_B1.line(ratio_arr[order], S_arr[order],
                 line_width=3, color=color)

plot_B1.output_backend = 'svg'
bokeh.io.show(plot_B1)

## Resource-competition onset across ribosome pool sizes

Activation curves for different total ribosome pools, with numerical and analytical estimates of the onset of resource competition.


In [7]:
# ============================================================
# Simulation settings
# ============================================================

t = np.linspace(0, 10, 100)

# Inputs
xv = np.linspace(0, 3, 300)
x2 = 0.5
x3 = 1.0

# Kinetic rates
ap1 = 10.0
ap2 = 10.0
ap3 = 10.0

p = 10.0
d = 1.0

d1 = d2 = d3 = 1.0
v1 = v2 = v3 = 1.0

b = 200.0
u = 1.0

# ============================================================
# Binding strengths
# ============================================================

e3 = 100.0
a3 = (d3 + v3) / e3

e_val = 1e-6
a1 = (d1 + v1) / e_val
a2 = (d2 + v2) / e_val

# ============================================================
# Resource pools
# ============================================================

rtot_values = [2.5, 1.5, 1.0]

colors = [
    '#000000',
    '#666666',
    '#BBBBBB'
]

# ============================================================
# Figure
# ============================================================

fig_size = [250, 225]

plot_rtot = bokeh.plotting.figure(
    width=fig_size[0],
    height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁'
)

plot_rtot.axis.major_label_text_font_size = '12pt'
plot_rtot.xaxis.axis_label_text_font_size = '14pt'
plot_rtot.yaxis.axis_label_text_font_size = '14pt'

# ============================================================
# Kink finder
# ============================================================

def find_kink(xv_grid, y_curve, x2):

    d2y = np.gradient(
        np.gradient(y_curve, xv_grid),
        xv_grid
    )

    mask = xv_grid > (x2 + 0.1)

    idx = np.where(mask)[0]

    return xv_grid[idx[np.argmin(d2y[idx])]]

# ============================================================
# Storage
# ============================================================

onset_num = []
onset_pred = []

# ============================================================
# Main sweep
# ============================================================

for rtot_val, color in zip(rtot_values, colors):

    y1_values = np.zeros_like(xv)

    # --------------------------------------------------------
    # Simulate activation curve
    # --------------------------------------------------------

    for i, xi in enumerate(xv):

        sol = scipy.integrate.odeint(
            Seq_TL_ODEs,
            np.zeros(10),
            t,
            args=(
                xi, x2, x3,
                ap1, ap2, ap3,
                p, d,
                a1, a2, a3,
                d1, d2, d3,
                v1, v2, v3,
                rtot_val,
                b,
                u
            )
        )

        y1_values[i] = sol[-1, 6]

    # --------------------------------------------------------
    # Plot activation function
    # --------------------------------------------------------

    plot_rtot.line(
        xv,
        y1_values,
        line_width=3,
        color=color
    )

    # --------------------------------------------------------
    # Numerical onset (kink)
    # --------------------------------------------------------

    x_num = find_kink(
        xv,
        y1_values,
        x2
    )

    y_num = np.interp(
        x_num,
        xv,
        y1_values
    )

    onset_num.append(x_num)

    plot_rtot.scatter(
        [x_num],
        [y_num],
        size=8,
        marker='circle',
        fill_color=color,
        line_color=color,
        line_width=1.5
    )

    # --------------------------------------------------------
    # Analytical prediction
    # --------------------------------------------------------

    x_pred = (
        rtot_val * (d + p)
        - ap2 * x2
    ) / ap1

    onset_pred.append(x_pred)

    plot_rtot.add_layout(
        bokeh.models.Span(
            location=x_pred,
            dimension='height',
            line_color=color,
            line_dash='dashed',
            line_width=2
        )
    )

plot_rtot.output_backend = "svg"

bokeh.io.show(plot_rtot)

## Numerical detection of onset shifts with ribosome supply

Activation curves for different total ribosome pools at fixed endogenous demand. Circles mark numerically detected onset locations.


In [8]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Inputs (finer x1 grid for accurate kink detection)
xv = np.linspace(0, 3, 300)
x2 = 0.5
x3 = 1.0   # fixed endogenous demand

# Kinetic rates
ap1 = 10.0              # (μM/hr)
ap2 = 10.0              # (μM/hr)
ap3 = 10.0              # (μM/hr)
p = 10.0                # hr⁻¹
d = 1.0                 # hr⁻¹
d1 = d2 = d3 = 1.0      # hr⁻¹
v1 = v2 = v3 = 1.0      # hr⁻¹
b = 200.0               # μM⁻¹·hr⁻¹
u = 1.0                 # hr⁻¹

# Endogenous species binding affinity (fixed, weak)
e3 = 100.0
a3 = (d3 + v3) / e3

# Strong binding for the synthetic circuit (sharp elbow)
e_val = 1e-6
a1 = (d1 + v1) / e_val
a2 = (d2 + v2) / e_val

# Sweep rtot
rtot_values = [1.0, 1.5, 2.5]
rtot_values = rtot_values[::-1]

# Grey gradient: dark = small pool (more competition), light = large pool
colors = ['#000000', '#666666', '#BBBBBB']

# Figure setup
fig_size = [250, 225]
plot_rtot = bokeh.plotting.figure(
    width=fig_size[0], height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁'
)
plot_rtot.axis.major_label_text_font_size = '12pt'
plot_rtot.xaxis.axis_label_text_font_size = '14pt'
plot_rtot.yaxis.axis_label_text_font_size = '14pt'

# Helper: locate kink numerically as point of most-negative second derivative,
# excluding the ReLU activation region near x2
def find_kink(xv_grid, y1_curve, x2):
    d2y = np.gradient(np.gradient(y1_curve, xv_grid), xv_grid)
    mask = xv_grid > (x2 + 0.1)
    idx = np.where(mask)[0]
    return xv_grid[idx[np.argmin(d2y[idx])]]


# For each rtot, compute the activation function and locate the kink
onset_x = []
onset_y = []

for rtot_val, color in zip(rtot_values, colors):

    y1_values = np.zeros_like(xv)

    for i, xi in enumerate(xv):
        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs,
            np.zeros(10),
            t,
            args=(xi, x2, x3,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot_val, b, u)
        )
        y1_values[i] = y_sol[-1, 6]

    plot_rtot.line(xv, y1_values, line_width=3, color=color)

    # Locate kink numerically and mark with a circle
    x1_kink = find_kink(xv, y1_values, x2)
    y_kink = float(np.interp(x1_kink, xv, y1_values))
    onset_x.append(x1_kink)
    onset_y.append(y_kink)
    plot_rtot.scatter([x1_kink], [y_kink],
                      size=8, marker='circle',
                      line_color = color,
                      fill_color=color,
                      line_width=1.5)

# Single arrow showing how the kink moves with increasing rtot
# if len(onset_x) >= 2:
#     arrow = Arrow(
#         end=OpenHead(size=10, line_color='red', line_width=2),
#         x_start=onset_x[0], y_start=onset_y[0],
#         x_end=onset_x[-1], y_end=onset_y[-1],
#         line_color='red', line_width=2
#     )
#     plot_rtot.add_layout(arrow)

plot_rtot.output_backend = 'svg'
bokeh.io.show(plot_rtot)

## Onset shifts with endogenous binding affinity and demand


In [9]:
# Simulation time (hours)
t = np.linspace(0, 10, 100)

# Inputs (finer x1 grid for accurate kink detection)
xv = np.linspace(0, 3, 300)
x2 = 0.5

# Kinetic rates
ap1 = 10.0              # (μM/hr)
ap2 = 10.0              # (μM/hr)
ap3 = 10.0              # (μM/hr)
p = 10.0                # hr⁻¹
d = 1.0                 # hr⁻¹
d1 = d2 = d3 = 1.0      # hr⁻¹
v1 = v2 = v3 = 1.0      # hr⁻¹
b = 200.0               # μM⁻¹·hr⁻¹
u = 1.0                 # hr⁻¹
rtot = 1.6             # μM

# Strong binding for the synthetic circuit (sharp elbow)
e_val = 1e-6
a1 = (d1 + v1) / e_val
a2 = (d2 + v2) / e_val

# Grey gradient: dark → light
colors = ['#000000', '#666666', '#BBBBBB']

# Helper: locate kink numerically
def find_kink(xv_grid, y1_curve, x2):
    d2y = np.gradient(np.gradient(y1_curve, xv_grid), xv_grid)
    mask = xv_grid > (x2 + 0.1)
    idx = np.where(mask)[0]
    return xv_grid[idx[np.argmin(d2y[idx])]]

# ----------------------------------------------------------------------------
# Panel L: sweep e3 with e3 <= e_val throughout
# (endogenous transcript binds at least as strongly as the synthetic ones)
#
# Note: under this constraint, the endogenous transcript truly competes
# even at synthetic saturation, so the kink DOES shift with e3.
# ----------------------------------------------------------------------------

# Fixed endogenous demand
x3 = 1.0

# Sweep e3, all values <= e_val (weakest → strongest endogenous binding)
# Ordering: dark = weakest of the strong (kink furthest right), light = strongest
e3_values = [1e-6, 1e-7, 1e-8]

fig_size = [250, 225]
plot_e3 = bokeh.plotting.figure(
    width=fig_size[0], height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁'
)
plot_e3.axis.major_label_text_font_size = '12pt'
plot_e3.xaxis.axis_label_text_font_size = '14pt'
plot_e3.yaxis.axis_label_text_font_size = '14pt'

onset_x_e3 = []
onset_y_e3 = []

for e3_val, color in zip(e3_values, colors):
    a3 = (d3 + v3) / e3_val
    y1_values = np.zeros_like(xv)
    for i, xi in enumerate(xv):
        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs,
            np.zeros(10),
            t,
            args=(xi, x2, x3,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot, b, u)
        )
        y1_values[i] = y_sol[-1, 6]

    plot_e3.line(xv, y1_values, line_width=3, color=color)

    # Locate kink numerically and mark with a circle
    x1_kink = find_kink(xv, y1_values, x2)
    y_kink = float(np.interp(x1_kink, xv, y1_values))
    onset_x_e3.append(x1_kink)
    onset_y_e3.append(y_kink)
    plot_e3.scatter([x1_kink], [y_kink],
                    size=12, marker='circle',
                    fill_color=color,
                    line_color=color, line_width=1.5)

# Single arrow showing how the kink moves as e3 strengthens
# if len(onset_x_e3) >= 2:
#     arrow_e3 = Arrow(
#         end=OpenHead(size=10, line_color='red', line_width=2),
#         x_start=onset_x_e3[0], y_start=onset_y_e3[0],
#         x_end=onset_x_e3[-1], y_end=onset_y_e3[-1],
#         line_color='red', line_width=2
#     )
#     plot_e3.add_layout(arrow_e3)

plot_e3.output_backend = 'svg'


# ----------------------------------------------------------------------------
# Panel R: sweep x3 with STRONG endogenous binding (kink DOES shift)
# ----------------------------------------------------------------------------

# Strong endogenous binding, matched to the synthetic circuit
e3 = e_val
a3 = (d3 + v3) / e3

# Sweep x3 (low → high endogenous demand). Kept small enough that the
# predicted kink stays above the ReLU threshold at x2 = 0.5.
x3_values = [0.1, 0.3, 0.6]

plot_x3 = bokeh.plotting.figure(
    width=fig_size[0], height=fig_size[1],
    x_axis_label='x₁',
    y_axis_label='y₁'
)
plot_x3.axis.major_label_text_font_size = '12pt'
plot_x3.xaxis.axis_label_text_font_size = '14pt'
plot_x3.yaxis.axis_label_text_font_size = '14pt'

onset_x = []
onset_y = []

for x3_val, color in zip(x3_values, colors):
    y1_values = np.zeros_like(xv)
    for i, xi in enumerate(xv):
        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs,
            np.zeros(10),
            t,
            args=(xi, x2, x3_val,
                  ap1, ap2, ap3,
                  p, d,
                  a1, a2, a3,
                  d1, d2, d3,
                  v1, v2, v3,
                  rtot, b, u)
        )
        y1_values[i] = y_sol[-1, 6]

    plot_x3.line(xv, y1_values, line_width=3, color=color)

    # Locate kink numerically and mark with a circle
    x1_kink = find_kink(xv, y1_values, x2)
    y_kink = float(np.interp(x1_kink, xv, y1_values))
    onset_x.append(x1_kink)
    onset_y.append(y_kink)
    plot_x3.scatter([x1_kink], [y_kink],
                    size=12, marker='circle',
                    fill_color=color,
                    line_color=color, line_width=1.5)

# Single arrow showing how the kink moves with increasing x3
# if len(onset_x) >= 2:
#     arrow = Arrow(
#         end=OpenHead(size=10, line_color='red', line_width=2),
#         x_start=onset_x[0], y_start=onset_y[0],
#         x_end=onset_x[-1], y_end=onset_y[-1],
#         line_color='red', line_width=2
#     )
#     plot_x3.add_layout(arrow)

plot_x3.output_backend = 'svg'

# Show both panels side by side
bokeh.io.show(bokeh.layouts.row(plot_e3, plot_x3))

## Ribosome allocation under heterogeneous binding affinities

Plot $S_1$ vs $S_2$ over the biologically meaningful range of $x_1$ (0 to 3). Three cases: $e_1 > e_2$, $e_1 = e_2$, $e_1 < e_2$. The diagonal corresponds to the homogeneous case; deviations above/below indicate which transcript captures more ribosomes under competition.


In [10]:
# Simulation parameters
N_bio = 50   # Points in biologically relevant range (0-3)
t = np.linspace(0, 10, 100)

# Biologically meaningful input range
xv_bio = np.linspace(0, 3, N_bio)

x2 = 0.5
x3 = 0.5

# Transcription rates (μM/hr)
ap1 = 10.0
ap2 = 10.0
ap3 = 10.0

# Degradation rates
p = 10.0
d = 1.0

# Ribosome binding (baseline)
a1 = a2 = a3 = 100.0
d1 = d2 = d3 = 1.0

# Translation
v1 = v2 = v3 = 1.0

# Sequestration
b = 200.0
u = 1.0

# Total ribosome pool
rtot = 1.0

# Base e values
e_base = (d1 + v1 + d + p) / (a1 * rtot)
e2_base = (d2 + v2 + d + p) / (a2 * rtot)

# Set up figure
fig_size = [250, 225]
plot_C = bokeh.plotting.figure(
    width=fig_size[0], height=fig_size[1],
    x_axis_label='S₂',
    y_axis_label='S₁',
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0)
)
plot_C.axis.major_label_text_font_size = '12pt'
plot_C.xaxis.axis_label_text_font_size = '12pt'
plot_C.yaxis.axis_label_text_font_size = '12pt'

# Helper function to compute S1 and S2 for given range
def sweep_S(a1_val, e1, xv_range):
    N = len(xv_range)
    S1_result = np.zeros(N)
    S2_result = np.zeros(N)

    for i in range(N):
        xi = xv_range[i]
        y_sol = scipy.integrate.odeint(
            Seq_TL_ODEs, np.zeros(10), t,
            args=(xi, x2, x3, ap1, ap2, ap3,
                  p, d, a1_val, a2, a3, d1, d2, d3,
                  v1, v2, v3, rtot, b, u))

        # Extract ribosome concentrations at steady state
        r1_ss = y_sol[-1, 3]
        r2_ss = y_sol[-1, 4]
        r3_ss = y_sol[-1, 5]

        # Compute free ribosomes (normalized)
        r_free = rtot - (r1_ss + r2_ss + r3_ss)
        r_n = r_free / rtot

        # Compute S1 and S2
        S1_result[i] = r_n / (r_n + e1) if (r_n + e1) > 0 else 0
        S2_result[i] = r_n / (r_n + e2_base) if (r_n + e2_base) > 0 else 0

    return S1_result, S2_result


# Define three cases
cases = [
    {"label": "e₁ > e₂", "factor": 1.5, "color": "#4C72B0"},   # blue
    {"label": "e₁ = e₂", "factor": 1.0, "color": "black"},
    {"label": "e₁ < e₂", "factor": 0.5, "color": "orange"},
]

for case in cases:
    e1 = e_base * case["factor"]
    a1_val = (d1 + v1 + d + p) / (e1 * rtot)

    # Biological range only (solid line, 0-3)
    S1_bio, S2_bio = sweep_S(a1_val, e1, xv_bio)

    plot_C.line(S2_bio, S1_bio,
                line_width=2.5,
                color=case["color"],
                legend_label=case["label"])

    # Mark start (x1=0) with circle, end (x1=3) with square
    plot_C.scatter([S2_bio[0]], [S1_bio[0]],
                   size=10, marker='circle',
                   fill_color=case["color"],
                   line_color='white', line_width=1.5)
    plot_C.scatter([S2_bio[-1]], [S1_bio[-1]],
                   size=10, marker='square',
                   fill_color=case["color"],
                   line_color='white', line_width=1.5)

# Diagonal reference line (S1 = S2)
plot_C.line([0, 1], [0, 1],
            line_width=1, color='gray', line_dash='dotted')

plot_C.legend.location = "bottom_right"
plot_C.legend.label_text_font_size = "10pt"
plot_C.output_backend = 'svg'
bokeh.io.show(plot_C)

Add the circles and the squares to relate to Panel E.